# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huzaifaguru/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

**Lane 4 — CTR / Engagement Opportunity Scoring** (provisional; carried from ML-02 and ML-03).

ML-03 framed the task on the 30k starter CSV and hit two walls I wrote down at the time: the
"windows" were slices of one aggregated row, and `avg_position` was a 90-day mean that overlapped the
target window. This notebook moves the contract onto the **warehouse daily table**, where I choose the
windows day by day — which fixes both.

> **Before Runtime → Run all:** request access on
> [`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)
> (instant), create a **plain Read** token in HF settings, and store it in Colab's 🔑 **Secrets** panel as
> `HF_TOKEN`. Never paste a token into a cell — this repo is public.
>
> Skills for this card: `skills/writing-data-contracts/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md`.
> Expect **~4–8 minutes** on the two aggregation cells; everything else is seconds.

**Month discipline.** `fact_content_daily_performance_sample` is *not* a random sample — it is exactly the
final month (June 2026), which is the natural outcome window of any past→future label. I develop on
**`month=2026-03`** as the feature window and **`month=2026-04`** as the outcome window, and I leave June
sealed as a test month I have not looked at.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Colab Secrets first, then environment, then a masked prompt. The token is never printed.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or getpass.getpass("HF READ token (hf_...): ")
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "No usable token found - set HF_TOKEN in Colab Secrets."
print("Token loaded (not shown).")

Token loaded (not shown).


In [3]:
import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

# My windows, chosen once and referenced everywhere below.
MONTH_FEATURES = "2026-03"   # feature window - mid-panel, well away from the sealed final month
MONTH_OUTCOME  = "2026-04"   # outcome window - strictly AFTER the feature window

FACT_FEAT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH_FEATURES}/*.parquet')"
FACT_OUT  = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH_OUTCOME}/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

# Setup smoke-test (NOT one of the three contract queries below): what columns actually exist?
schema = con.sql(f"DESCRIBE SELECT * FROM {FACT_FEAT} LIMIT 1").df()
print(f"fact_content_daily_performance, month={MONTH_FEATURES} - {len(schema)} columns\n")
print(", ".join(schema["column_name"].tolist()))

fact_content_daily_performance, month=2026-03 - 31 columns

report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events, month


## 1. Unit of analysis + time window

### The contract, in five plain answers

**1. What one row means for my lane.**
In the *source* table, one row = **one content item, for one client, on one day** — grain is
`report_date + client_hash_id + content_hash_id`. In my *feature frame*, one row =
**one content item (a page), aggregated over the whole feature month**. The page is the unit because the
page is what a reviewer opens and edits. The grain follows the action.

**2. Which tables I use.**
`fact_content_daily_performance` — the two month partitions below, and nothing else. I skip
`fact_content_query_90d` on purpose for now: its window is a fixed trailing 90 days that overlaps my
outcome month, so joining it today would import my label into my features. It comes back in a later card
once I can line the windows up. `dim_clients` is read for context only (history coverage), never joined
into features.

**3. Which time window.**

```text
FEATURES                       OUTCOME
month=2026-03                  month=2026-04
2026-03-01 .. 2026-03-31  ->   2026-04-01 .. 2026-04-30
(everything a model may see)   (the thing it predicts)
```

The two windows do not touch at any point. That single property is what the whole notebook exists to
protect, and section 3's trap shows what happens the moment it is broken.

**4. What I would predict or rank (label / proxy).**
`under_next30` — **1 when a page's click-through rate in April falls in the bottom quartile of pages with
comparable April impression volume**, among pages with ≥ 500 impressions in *both* months. The clicks and
impressions are observed; the quartile cut, the peer group and the 500-impression floor are policy choices
I own. So: a **proxy**, and I will keep calling it that.

**5. One thing I deliberately exclude.**
**`gsc_clicks` and `gsc_impressions` from the outcome month, in every form.** They are the label. The
temptation is real because they are right there in the same table under the same names — which is exactly
why section 3 adds one of them on purpose, measures the damage, and removes it.

*(Runner-up exclusion, for the same reason as ML-03: any position figure averaged over a window that
overlaps April. Here I compute position inside March only — the warehouse fixes the leak the starter CSV
forced on me.)*

## 2. Fields: feature / label / context / excluded

Every column I may touch, in exactly one bucket. Excluded gets a why.

| Column | Bucket | Why |
|---|---|---|
| `report_date` | context | Defines the windows; never fed to a model |
| `client_hash_id` | context | Pseudonym. **Grouped-split key.** Never a feature — a model that learns the client has learned nothing |
| `content_hash_id` | context | Pseudonym. Row key and join key only |
| `gsc_impressions` (March) | **feature** | Exposure the page had before the decision moment |
| `gsc_clicks` (March) | **feature** | Past outcome. The past is allowed to be a feature; the future is not |
| `gsc_avg_position` (March) | **feature** | Impression-weighted over March days only — the ML-03 leak, fixed |
| days with impressions (March) | **feature** | Derived from March rows only |
| CTR percentile vs March peers | **feature** | Derived from March columns only |
| `gsc_impressions` (April) | label-side | Eligibility filter + peer group for the label. Not a free feature |
| `gsc_clicks` (April) | **label** | The outcome being predicted |
| `under_next30` | **label** | The proxy defined in section 1 |
| `ga4_data_available` | context | Availability flag — decides which rows are trustworthy, checked in Query 3 |
| GA4 columns (`sessions`, engagement, scroll) | **excluded** | Zero-*filled*, not zero-*measured*, before a client's `ga4_data_start`. Section 4 quantifies how much of the panel that affects |
| `sessions_ai` | **excluded** | Sparse to the point of uselessness at this grain — 30,177 rows with AI sessions against 78.8M daily rows |
| `fact_content_query_90d.*` | **excluded** | Fixed 90-day window overlaps April. Window alignment first, join later |

The rule underneath all of it: **a feature must be knowable at the decision moment.** The decision moment
here is *the last second of 2026-03-31*. If a column could not have been computed then, it is not a
feature, however useful it looks.

## 3. Verify it with queries (grain, counts, missing values, windows)

Three queries, one per contract claim. A contract line without a query next to it is a guess.

### Query 1 of 3 — the grain really is what I said

In [4]:
grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT_FEAT}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows where (report_date, client_hash_id, content_hash_id) repeats: {len(grain)}")
print("Zero rows back => the grain holds: one row = one content item, one client, one day.\n")
grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where (report_date, client_hash_id, content_hash_id) repeats: 0
Zero rows back => the grain holds: one row = one content item, one client, one day.



,report_date,client_hash_id,content_hash_id,n


### Query 2 of 3 — my slice's row count and date span

In [5]:
span = con.sql(f"""
    SELECT COUNT(*)                          AS rows_in_month,
           COUNT(DISTINCT content_hash_id)   AS content_items,
           COUNT(DISTINCT client_hash_id)    AS clients,
           MIN(report_date)                  AS first_day,
           MAX(report_date)                  AS last_day,
           COUNT(DISTINCT report_date)       AS distinct_days
    FROM {FACT_FEAT}
""").df()

print(f"Feature window: month={MONTH_FEATURES}")
print(span.to_string(index=False))
print()
print("Contract claim being checked: the feature window is exactly March 2026 and nothing else.")
print("If first_day/last_day fall outside 2026-03, my partition path is wrong and every")
print("downstream number is wrong with it.")

Feature window: month=2026-03
 rows_in_month  content_items  clients  first_day   last_day  distinct_days
       9841378         331437       55 2026-03-01 2026-03-31             31

Contract claim being checked: the feature window is exactly March 2026 and nothing else.
If first_day/last_day fall outside 2026-03, my partition path is wrong and every
downstream number is wrong with it.


### Query 3 of 3 — availability, checked with `IS TRUE`

`ga4_data_available = FALSE` marks rows where the GA4 columns were **zero-filled because tracking had not
started**, not measured as zero. Reading those zeros as "no engagement" is the single most expensive
mistake available in this table, so I check how many rows survive the flag before I trust any GA4 column.

`IS TRUE` rather than `= TRUE` on purpose: `IS TRUE` returns false for NULL instead of returning NULL, so
rows where the flag itself is missing fall on the safe side of the filter rather than vanishing silently.

In [6]:
avail = con.sql(f"""
    SELECT COUNT(*)                                                   AS all_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)         AS ga4_available_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE)     AS ga4_not_available_rows,
           COUNT(*) FILTER (WHERE gsc_impressions > 0)                AS rows_with_impressions,
           COUNT(*) FILTER (WHERE gsc_impressions > 0
                              AND ga4_data_available IS TRUE)         AS both
    FROM {FACT_FEAT}
""").df()

r = avail.iloc[0]
print(f"month={MONTH_FEATURES}")
print(f"  all rows                          : {r.all_rows:,}")
print(f"  ga4_data_available IS TRUE        : {r.ga4_available_rows:,} "
      f"({r.ga4_available_rows/r.all_rows*100:.1f}% survive)")
print(f"  NOT TRUE (zero-filled or missing) : {r.ga4_not_available_rows:,} "
      f"({r.ga4_not_available_rows/r.all_rows*100:.1f}%)")
print(f"  rows with any search impression   : {r.rows_with_impressions:,} "
      f"({r.rows_with_impressions/r.all_rows*100:.1f}%)")
print(f"  both conditions                   : {r.both:,}")
print()
print("This is why my feature set is GSC-only. Whatever share of rows fails the GA4 flag is a share")
print("of the panel where an engagement feature would be silently fabricated from filler zeros.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

month=2026-03
  all rows                          : 9,841,378
  ga4_data_available IS TRUE        : 413,966 (4.2% survive)
  NOT TRUE (zero-filled or missing) : 9,427,412 (95.8%)
  rows with any search impression   : 3,611,061 (36.7%)
  both conditions                   : 364,347

This is why my feature set is GSC-only. Whatever share of rows fails the GA4 flag is a share
of the panel where an engagement feature would be silently fabricated from filler zeros.


## Five features, max — and one line each on *why it is knowable at the decision moment*

The decision moment is **the last second of 2026-03-31**. Every feature below is computed only from rows
whose `report_date` falls inside March, so every one of them could have been computed at that instant.

| # | Feature | Knowable at the decision moment because… |
|---|---|---|
| 1 | `imp_mar` — March impressions | It is a sum over March days only; on 31 March it is finished and final. Exposure the page *already had*. |
| 2 | `ctr_mar` — March clicks ÷ March impressions | Both inputs are March-only sums. It describes what already happened, not what happens next. |
| 3 | `ctr_pctile_mar` — CTR rank vs same-volume March peers | A ranking computed *within the March frame*: every page it compares against is also fully measured by 31 March. |
| 4 | `active_days_mar` — days in March with ≥ 1 impression | Counted from March rows only. Says whether exposure was steady or a one-day spike — a page that ranked for three days is not the same animal as one that ranked for thirty. |
| 5 | `pos_mar` — impression-weighted average position across March | **This is the ML-03 fix.** In the starter CSV `avg_position` was a 90-day mean that swallowed the outcome window. Here I compute it from March days only, so it is legitimately knowable on 31 March. |

Five, and no more, on purpose: this card is about proving the contract holds, not about winning ML-08.

**The heavy cell.** It scans two month partitions and returns one small row per page. Expect a few minutes.

In [7]:
feat = con.sql(f"""
    WITH mar AS (
        SELECT client_hash_id,
               content_hash_id,
               SUM(gsc_impressions)                                    AS imp_mar,
               SUM(gsc_clicks)                                         AS clk_mar,
               COUNT(*) FILTER (WHERE gsc_impressions > 0)             AS active_days_mar,
               SUM(gsc_avg_position * gsc_impressions)
                   FILTER (WHERE gsc_impressions > 0 AND gsc_avg_position > 0)
                 / NULLIF(SUM(gsc_impressions)
                   FILTER (WHERE gsc_impressions > 0 AND gsc_avg_position > 0), 0) AS pos_mar
        FROM {FACT_FEAT}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 500
    ),
    apr AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS imp_apr,
               SUM(gsc_clicks)      AS clk_apr
        FROM {FACT_OUT}
        GROUP BY 1
        HAVING SUM(gsc_impressions) >= 500
    )
    SELECT m.client_hash_id, m.content_hash_id,
           m.imp_mar, m.clk_mar, m.active_days_mar, m.pos_mar,
           a.imp_apr, a.clk_apr
    FROM mar m
    JOIN apr a ON a.content_hash_id = m.content_hash_id
""").df()

print(f"pages eligible in BOTH windows (>=500 impressions each): {len(feat):,}")
print(f"clients represented                                   : {feat['client_hash_id'].nunique()}")
assert feat["content_hash_id"].is_unique, "grain broken: feature frame must be one row per page"
print("Feature-frame grain check passed: one row = one content item.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages eligible in BOTH windows (>=500 impressions each): 51,496
clients represented                                   : 34
Feature-frame grain check passed: one row = one content item.


In [8]:
MIN_IMPRESSIONS = 500   # policy: below this, CTR is noise (carried from ML-03, re-justified there)
BOTTOM_QUANTILE = 0.25  # policy: "under-capturing" = worst quarter of its volume peer group
N_BANDS         = 4     # policy: peer group = quartiles of outcome-month impressions

d = feat.copy()

# ---- THE FIVE FEATURES (March only) ---------------------------------------
d["ctr_mar"] = d["clk_mar"] / d["imp_mar"] * 100
d["band_mar"] = pd.qcut(d["imp_mar"], N_BANDS, labels=False, duplicates="drop")
d["ctr_pctile_mar"] = d.groupby("band_mar", observed=True)["ctr_mar"].rank(pct=True)
FEATURES = ["imp_mar", "ctr_mar", "ctr_pctile_mar", "active_days_mar", "pos_mar"]

# ---- THE LABEL (April only) ------------------------------------------------
d["ctr_apr"] = d["clk_apr"] / d["imp_apr"] * 100
d["band_apr"] = pd.qcut(d["imp_apr"], N_BANDS, labels=False, duplicates="drop")
d["under_next30"] = (d.groupby("band_apr", observed=True)["ctr_apr"]
                       .rank(pct=True) <= BOTTOM_QUANTILE).astype(int)

BASE_RATE = d["under_next30"].mean()
print(f"pages: {len(d):,}   clients: {d['client_hash_id'].nunique()}")
print(f"positives: {int(d['under_next30'].sum()):,}   BASE RATE: {BASE_RATE:.3f}")
print()
print("Label rate per April volume band (should be ~0.25 everywhere):")
print(d.groupby("band_apr", observed=True)["under_next30"].mean().round(3).to_string())
print()
print("The five features, and the two label-side columns kept only for the trap below:")
d[["content_hash_id"] + FEATURES + ["ctr_apr", "under_next30"]].head(8)

pages: 51,496   clients: 34
positives: 15,121   BASE RATE: 0.294

Label rate per April volume band (should be ~0.25 everywhere):
band_apr
0    0.425
1    0.250
2    0.250
3    0.250

The five features, and the two label-side columns kept only for the trap below:


,content_hash_id,imp_mar,ctr_mar,ctr_pctile_mar,active_days_mar,pos_mar,ctr_apr,under_next30
0,content_2e6360ad20fd7107,899.0,0.111235,0.376746,31,5.924208,0.000000,1
1,content_65c50dfe9d87a585,3108.0,0.000000,0.027702,30,6.953668,0.065232,0
2,content_cdd114d71966c437,1888.0,0.000000,0.069453,30,10.622881,0.138696,0
3,content_26f5092ee7f70d45,4314.0,0.000000,0.027702,31,7.894298,0.000000,1
4,content_c7db213441834978,1965.0,0.610687,0.869096,29,5.789313,0.320712,0
5,content_9e7c70abfbae371e,2436.0,0.000000,0.027702,31,5.726601,0.000000,1
6,content_0a03168a23b99875,1429.0,0.139958,0.405764,30,3.456263,0.000000,1
7,content_a633d5b26395212e,2047.0,0.244260,0.587049,29,3.569126,0.109290,0


## The trap — one label-derived column, on purpose

The lesson from notebook 02, performed on real warehouse data. I add **exactly one** column that has no
business being a feature — `ctr_apr`, the April click-through rate — and watch what a quick model does with
it.

`ctr_apr` is not obviously poisonous. It is a normal-looking column, in the same dataframe, with a name one
character away from `ctr_mar`. That is the whole point: leakage does not arrive labelled.

Validation design (fixed before any number appears, so I cannot rationalise afterwards): **grouped split on
`client_hash_id`** — pages from a client never appear on both sides — reporting ROC-AUC and precision@50.

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

model_df = d.dropna(subset=FEATURES).reset_index(drop=True)
groups = model_df["client_hash_id"]
y = model_df["under_next30"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, test_idx = next(splitter.split(model_df, y, groups))
print(f"train {len(train_idx):,} pages / {groups.iloc[train_idx].nunique()} clients   |   "
      f"test {len(test_idx):,} pages / {groups.iloc[test_idx].nunique()} clients   |   "
      f"clients shared: {len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))}")

def quick_score(cols, label):
    X = model_df[cols]
    m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    m.fit(X.iloc[train_idx], y.iloc[train_idx])
    p = m.predict_proba(X.iloc[test_idx])[:, 1]
    auc = roc_auc_score(y.iloc[test_idx], p)
    order = np.argsort(-p)[:50]
    p50 = y.iloc[test_idx].to_numpy()[order].mean()
    print(f"  {label:34s} ROC-AUC = {auc:.3f}   precision@50 = {p50:.3f}")
    return auc, p50, m

print(f"\n  {'base rate (random shortlist)':34s} ROC-AUC = 0.500   precision@50 = {BASE_RATE:.3f}")
honest_auc, honest_p50, honest_model = quick_score(FEATURES, "HONEST: the five features")

train 25,321 pages / 23 clients   |   test 26,175 pages / 11 clients   |   clients shared: 0

  base rate (random shortlist)       ROC-AUC = 0.500   precision@50 = 0.294
  HONEST: the five features          ROC-AUC = 0.802   precision@50 = 0.920


In [10]:
LEAK = "ctr_apr"   # April CTR - the label is a quantile of this exact column
leak_auc, leak_p50, leak_model = quick_score(FEATURES + [LEAK], f"LEAKED: + {LEAK}")

print()
print(f"ROC-AUC       {honest_auc:.3f}  ->  {leak_auc:.3f}   (+{leak_auc-honest_auc:.3f})")
print(f"precision@50  {honest_p50:.3f}  ->  {leak_p50:.3f}   (+{leak_p50-honest_p50:.3f})")
print()
imp = (pd.Series(leak_model.feature_importances_, index=FEATURES + [LEAK])
         .sort_values(ascending=False))
print("Where the leaked model's attention went:")
print(imp.round(3).to_string())
print()
print(f"'{LEAK}' takes {imp[LEAK]*100:.0f}% of the importance - more than any real feature.")
print("The model did not learn anything about search behaviour; it learned to read the answer key.")
print("On 31 March this column did not exist.")
print()
if (leak_auc - honest_auc) < 0.02 and (leak_p50 - honest_p50) < 0.02:
    print("NOTE: the headline scores barely moved this time. That does NOT mean the column was")
    print("harmless - the importance table above is the tell. A leak that does not move the metric")
    print("is more dangerous, not less, because nothing warns you. Judge leakage by whether the")
    print("column could have been known at the decision moment, never by whether the score jumped.")

  LEAKED: + ctr_apr                  ROC-AUC = 0.999   precision@50 = 1.000

ROC-AUC       0.802  ->  0.999   (+0.197)
precision@50  0.920  ->  1.000   (+0.080)

Where the leaked model's attention went:
ctr_apr            0.800
ctr_mar            0.088
ctr_pctile_mar     0.058
imp_mar            0.031
pos_mar            0.018
active_days_mar    0.005

'ctr_apr' takes 80% of the importance - more than any real feature.
The model did not learn anything about search behaviour; it learned to read the answer key.
On 31 March this column did not exist.



In [11]:
# Delete it. Not comment it out - delete it, so it cannot come back by accident.
d = d.drop(columns=[LEAK], errors="ignore")
model_df = model_df.drop(columns=[LEAK], errors="ignore")

assert LEAK not in model_df.columns, "the leaked column is still here"
assert not any(c.endswith("_apr") for c in FEATURES), "an outcome-window column got into FEATURES"
print(f"Dropped '{LEAK}'. Guard passed: no outcome-window column remains in the feature set.\n")

print("THE NUMBER I KEEP - the five honest features, grouped split on client_hash_id:")
print(f"  ROC-AUC      = {honest_auc:.3f}")
print(f"  precision@50 = {honest_p50:.3f}   (base rate {BASE_RATE:.3f}, "
      f"lift {honest_p50/BASE_RATE:.2f}x)")
print()
print("Lower than the leaked run, and it is the only one that means anything. A model cannot")
print("know April's CTR in March; a notebook that lets it is measuring nothing.")

Dropped 'ctr_apr'. Guard passed: no outcome-window column remains in the feature set.

THE NUMBER I KEEP - the five honest features, grouped split on client_hash_id:
  ROC-AUC      = 0.802
  precision@50 = 0.920   (base rate 0.294, lift 3.13x)

Lower than the leaked run, and it is the only one that means anything. A model cannot
know April's CTR in March; a notebook that lets it is measuring nothing.


## 4. Data limits — what this slice can never tell me

**The named limitation of my slice: it is one calendar month of one unbalanced panel, and every client
in it is compared against the same calendar boundary.**

The history depth differs wildly per client — `dim_clients.gsc_data_start` is the honest per-client start
date, and a global "March 2026" window quietly treats a client with 17 months of history and a client
whose tracking began in February as the same kind of evidence. The query below measures how much of my
eligible panel is actually affected rather than leaving it as a worry.

In [12]:
cl = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {DIM_CLIENTS}
""").df()
cl["gsc_data_start"] = pd.to_datetime(cl["gsc_data_start"])

mine = cl[cl["client_hash_id"].isin(model_df["client_hash_id"].unique())].copy()
cutoff = pd.Timestamp(f"{MONTH_FEATURES}-01")
mine["months_before_window"] = ((cutoff - mine["gsc_data_start"]).dt.days / 30.44).round(1)

print(f"Clients in my eligible panel: {len(mine)}")
print(f"  history before my feature window (months): "
      f"min {mine['months_before_window'].min():.1f}, "
      f"median {mine['months_before_window'].median():.1f}, "
      f"max {mine['months_before_window'].max():.1f}")
print(f"  clients with LESS than 3 months of history before 2026-03-01: "
      f"{int((mine['months_before_window'] < 3).sum())}")
print(f"  clients whose GSC history starts AFTER my window opens: "
      f"{int((mine['gsc_data_start'] >= cutoff).sum())}")
print()
print("Whatever that count is, those clients are thinner evidence than the rest, and my")
print("grouped split can put a whole thin client on the test side. That is a real source of")
print("variance in every number above - which is why one split is a starting point, not a result.")

Clients in my eligible panel: 34
  history before my feature window (months): min -0.9, median 4.4, max 13.1
  clients with LESS than 3 months of history before 2026-03-01: 13
  clients whose GSC history starts AFTER my window opens: 3

Whatever that count is, those clients are thinner evidence than the rest, and my
grouped split can put a whole thin client on the test side. That is a real source of
variance in every number above - which is why one split is a starting point, not a result.


### The rest of the honest list

- **One month is not a trend.** March → April is a single hop. It cannot distinguish a real, sustained
  decline from seasonality, a SERP-layout change, or a one-month blip — the decline-vs-consolidation-vs-
  noise test needs several consecutive windows, and I have not run it.
- **No consolidation check yet.** If a sibling page absorbed a page's clicks, this contract sees an
  under-capturing page and says nothing about where the demand went. That needs `url_hash_id` /
  `keyword_hash_id` grouping, which is a later card.
- **GA4 is out of scope by measurement, not by choice.** Query 3 quantifies how many rows fail
  `ga4_data_available IS TRUE`; on those rows the engagement columns are filler zeros. The engagement half
  of Lane 4 stays parked until I can filter on the flag and still have a panel worth modelling.
- **`sessions_ai` is unusable at this grain** — 30,177 rows carry AI sessions against 78.8M daily rows.
  Any classifier built on that would look impressive and mean nothing.
- **Nothing here is causal.** I can say a page under-captured relative to its peers. I cannot say a title
  rewrite would fix it — that needs an experiment this panel cannot provide.
- **One split, one seed.** Every number in section 3 comes from a single `GroupShuffleSplit`. Repeated
  splits and a spread come in ML-09; until then these are directional.

### What the contract hands to a human, in one sentence
> A ranked list of pages that under-captured clicks relative to same-volume peers in the month after the
> one their features come from, each with the March evidence behind it, for a reviewer to open in order
> and decide what — if anything — is actually wrong.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — pseudonymized hash ids only, and the HF token
      is read from Colab Secrets, never typed into a cell
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Card checklist, line by line
- [x] **Five plain-words contract answers** — section 1
- [x] **Exactly three verification queries with outputs visible** — Query 1 (grain), Query 2 (row count +
      date span), Query 3 (availability via `IS TRUE`)
- [x] **Five features, each with an "available when?" line** — the table above section 3's heavy cell
- [x] **The deliberate leak shown and removed** — `ctr_apr` added, scored, importance inspected, dropped,
      with an assert so it cannot return
- [x] **One named limitation** — the unbalanced panel against a global calendar window, measured against
      `dim_clients.gsc_data_start`

### What I owe ML-05 / ML-06
1. Re-run this contract across **three consecutive month pairs**, not one, and report the spread.
2. Bring in `fact_content_query_90d` **only after** aligning its fixed 90-day window against my outcome
   month — query-mix (how many distinct queries a page ranks for, how concentrated) is the most promising
   unused signal I have, and the most dangerous to join carelessly.
3. Sensitivity pass on all three policy thresholds: 500 impressions, bottom quartile, 4 volume bands.
4. Repeated grouped splits, so "precision@50" stops being one number and becomes a range.